# Count Models: Geometric, Hypergeometric, And Poisson

Official MA1001B alignment: 2.4 geometric and negative binomial; 2.5 hypergeometric; 2.6 Poisson; 2.7 data science links.


## How To Use This Lesson

Read the explanation cells before running the code. Run each code cell in order. When a checkpoint appears, stop and write your answer before continuing. The goal is not only to obtain output; the goal is to justify a decision from data.


## Learning Goals

- Explain the statistical idea in words.
- Implement the idea in Python with readable code.
- Interpret the result as evidence for a decision.
- State at least one assumption or limitation.


## Decision Scenario

A mobility planner wants to model hourly bike demand. A simple count model may help, but only if its assumptions are reasonable.


## Conceptual Explanation

Count models describe nonnegative integer outcomes. A Poisson model is useful for counts in a fixed interval when events occur independently at a roughly constant rate. Real data often violate that constant-rate assumption because time, weather, and context change demand.


## Mathematical Anchor

If X follows Poisson(lambda), then P(X = k) = exp(-lambda) lambda^k / k!, and E[X] = Var(X) = lambda.


## Data And Workflow Notes

Uses Bike Sharing Demand if available; otherwise simulates counts with daily variation.


## Python Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## Worked Example


In [ ]:
path = Path("data/raw/bike-sharing/train.csv")
if path.exists():
    bike = pd.read_csv(path)
    counts = bike["count"].dropna()
else:
    print(f"Missing {path}. Download Bike Sharing Demand from https://www.kaggle.com/c/bike-sharing-demand")
    hour = np.tile(np.arange(24), 60)
    rate = 40 + 90 * ((hour >= 7) & (hour <= 9)) + 80 * ((hour >= 17) & (hour <= 19))
    counts = pd.Series(rng.poisson(rate), name="count")

counts.describe()


In [ ]:
pd.Series({
    "mean": counts.mean(),
    "variance": counts.var(ddof=1),
    "variance_to_mean_ratio": counts.var(ddof=1) / counts.mean(),
}).round(2)


## From Calculation To Evidence


In [ ]:
ax = sns.histplot(counts, bins=30)
ax.set_title("Observed count distribution")
ax.set_xlabel("Count")
plt.show()


In [ ]:
lambda_hat = counts.mean()
threshold = counts.quantile(0.90)
pd.Series({
    "lambda_hat": lambda_hat,
    "observed_P_above_90th_percentile": (counts >= threshold).mean(),
    "poisson_P_above_same_threshold": stats.poisson.sf(threshold - 1, lambda_hat),
}).round(3)


## Guided Checkpoint

Use the variance-to-mean ratio to decide whether a single Poisson model is plausible.


## Common Mistakes

- Assuming all count data are automatically Poisson.
- Ignoring time segmentation when the rate clearly changes.
- Comparing only means without checking variability.


## Independent Practice

Create two segments, such as high-demand and low-demand hours. Compare the mean, variance, and Poisson tail probability for each segment.


## Interpretation Template

Use this structure for your written answer:

1. The decision question is ...
2. The statistical evidence is ...
3. The uncertainty or limitation is ...
4. Therefore, I recommend ... because ...


## Exit Ticket

What does overdispersion mean in practical planning language?
